# Modified TPC_RP Algorithm — CIFAR-10
## Task 3: Algorithm Modification — Dynamic Neighbor Scaling (DNS)

**Modification:** Replace the fixed K=20 in the typicality calculation with K = √(cluster_size), adapting dynamically to each cluster.

**Justification:** In the low-budget regime, clusters contain 10–50 points. Using K=20 in a small cluster means looking at nearly every point, making the density estimate noisy. DNS adapts K to each cluster's size, providing more focused density estimates in small clusters and more robust estimates in large ones.


---
## 1. Setup: Install Libraries & Imports


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import resnet18

import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
import random

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


---
## 2. Load CIFAR-10 Dataset


In [ ]:
basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=basic_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=basic_transform)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")


---
## 3. Load Pre-trained SimCLR Features


In [ ]:
features = np.load('/content/drive/MyDrive/Colab Notebooks/simclr_features_500epochs.npy')
print(f"Loaded train features: {features.shape}")

train_labels = np.array(train_dataset.targets)
test_labels = np.array(test_dataset.targets)


---
## 4. Typicality Score

Same as original — the function accepts K as a parameter.


In [ ]:
def compute_typicality(features, k_neighbors=20):
    nn_model = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    nn_model.fit(features)
    distances, _ = nn_model.kneighbors(features)
    distances = distances[:, 1:]
    avg_distances = distances.mean(axis=1)
    typicality = 1.0 / avg_distances
    return typicality


---
## 5. Modified Selection Function — DNS

The **only change** to the algorithm: instead of `k = min(20, cluster_size - 1)`,
we use `k = min(sqrt(cluster_size), cluster_size - 1)` with a minimum of 2.


In [ ]:
# MODIFIED TPC_RP selection with Dynamic Neighbor Scaling (DNS)
def typicality_clustering_select_dns(features, budget, labeled_indices=None, max_clusters=500):
    if labeled_indices is None:
        labeled_indices = []

    labeled_set = set(labeled_indices)
    n_labeled = len(labeled_indices)
    n_clusters = min(n_labeled + budget, max_clusters)

    print(f"  Running K-Means with {n_clusters} clusters...")
    if n_clusters <= 50:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    else:
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)

    cluster_assignments = kmeans.fit_predict(features)

    cluster_label_counts = {}
    for idx in labeled_indices:
        c = cluster_assignments[idx]
        cluster_label_counts[c] = cluster_label_counts.get(c, 0) + 1

    cluster_to_indices = {}
    for idx in range(len(features)):
        c = cluster_assignments[idx]
        if c not in cluster_to_indices:
            cluster_to_indices[c] = []
        cluster_to_indices[c].append(idx)

    selected = []

    for _ in range(budget):
        best_cluster = None
        best_cluster_fewest_labels = float('inf')
        best_cluster_size = -1

        for c, indices in cluster_to_indices.items():
            if len(indices) <= 5:
                continue
            n_labeled_in_cluster = cluster_label_counts.get(c, 0)
            if (n_labeled_in_cluster < best_cluster_fewest_labels or
                (n_labeled_in_cluster == best_cluster_fewest_labels and
                 len(indices) > best_cluster_size)):
                best_cluster = c
                best_cluster_fewest_labels = n_labeled_in_cluster
                best_cluster_size = len(indices)

        if best_cluster is None:
            break

        cluster_indices = cluster_to_indices[best_cluster]
        cluster_features = features[cluster_indices]

        # DNS: K = sqrt(cluster_size), minimum 2
        k = min(int(np.sqrt(len(cluster_indices))), len(cluster_indices) - 1)
        k = max(k, 2)

        cluster_typicality = compute_typicality(cluster_features, k_neighbors=k)
        sorted_by_typicality = np.argsort(-cluster_typicality)
        for rank in sorted_by_typicality:
            idx = cluster_indices[rank]
            if idx not in labeled_set and idx not in selected:
                selected.append(idx)
                break

        cluster_label_counts[best_cluster] = cluster_label_counts.get(best_cluster, 0) + 1

    return selected

print("DNS selection function ready")


---
## 6. Framework (i): Fully Supervised — Train Classifier Function


In [ ]:
def train_classifier(labeled_indices, train_dataset, test_dataset, device,
                     num_epochs=100, lr=0.025, batch_size=64):
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])
    train_data_augmented = datasets.CIFAR10(root='./data', train=True, download=False,
                                            transform=train_transform)
    labeled_subset = Subset(train_data_augmented, labeled_indices)
    actual_batch_size = min(batch_size, len(labeled_indices))
    train_loader = DataLoader(labeled_subset, batch_size=actual_batch_size,
                              shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

    model = resnet18(weights=None, num_classes=10).to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100.0 * correct / total

print("Classifier ready")


---
## 7. Framework (i): DNS Active Learning Loop

**Run this on IBEX** — takes hours with GPU.


In [ ]:
# Framework (i) — DNS Modified (RUN ON IBEX)
BUDGET_PER_ROUND = 10
NUM_ROUNDS = 5
NUM_REPETITIONS = 10

dns_fw1_all = []

for rep in range(NUM_REPETITIONS):
    set_seed(rep)
    print(f"\nDNS Framework(i) — Rep {rep+1}/{NUM_REPETITIONS}")
    labeled_indices = []
    round_accs = []

    for round_num in range(NUM_ROUNDS):
        new_indices = typicality_clustering_select_dns(
            features=features, budget=BUDGET_PER_ROUND,
            labeled_indices=labeled_indices, max_clusters=500
        )
        labeled_indices.extend(new_indices)
        acc = train_classifier(labeled_indices, train_dataset, test_dataset, device)
        round_accs.append(acc)
        print(f"  Round {round_num+1}: Budget={len(labeled_indices)}, Acc={acc:.2f}%")

    dns_fw1_all.append(round_accs)

dns_fw1_all = np.array(dns_fw1_all)
dns_fw1_mean = dns_fw1_all.mean(axis=0)
dns_fw1_std = dns_fw1_all.std(axis=0)

print(f"\nDNS Framework (i) Results:")
for i in range(NUM_ROUNDS):
    b = (i+1) * BUDGET_PER_ROUND
    print(f"Budget {b:4d}: {dns_fw1_mean[i]:.2f}% +/- {dns_fw1_std[i]:.2f}%")


---
## 8. Framework (ii): Linear Classifier on SimCLR Embeddings

**Run this on Colab** — fast, takes ~15 minutes.


In [ ]:
# Load SimCLR model and extract test features
class SimCLRModel(nn.Module):
    def __init__(self, feature_dim=128):
        super(SimCLRModel, self).__init__()
        backbone = resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        self.projection_head = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, feature_dim)
        )

    def forward(self, x):
        features = self.encoder(x)
        features = features.squeeze()
        projections = self.projection_head(features)
        return features, projections

model_simclr = SimCLRModel(feature_dim=128).to(device)
model_simclr.load_state_dict(torch.load(
    '/content/drive/MyDrive/Colab Notebooks/simclr_model_500epochs.pth',
    map_location=device))
model_simclr.eval()

@torch.no_grad()
def extract_test_features(model, device, batch_size=256):
    test_feature_dataset = datasets.CIFAR10(
        root='./data', train=False, download=False, transform=basic_transform)
    loader = DataLoader(test_feature_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    all_features = []
    for images, _ in loader:
        images = images.to(device)
        feats, _ = model(images)
        feats = F.normalize(feats, dim=1)
        all_features.append(feats.cpu().numpy())
    return np.concatenate(all_features, axis=0)

test_features = extract_test_features(model_simclr, device)
print(f"Test features: {test_features.shape}")

# Framework (ii) linear classifier
def train_linear_classifier(labeled_indices, train_features, train_labels,
                            test_features, test_labels, device,
                            num_epochs=200, lr=2.5, batch_size=64):
    feature_dim = train_features.shape[1]
    num_classes = len(np.unique(train_labels))
    X_train = torch.tensor(train_features[labeled_indices], dtype=torch.float32)
    y_train = torch.tensor(train_labels[labeled_indices], dtype=torch.long)
    X_test = torch.tensor(test_features, dtype=torch.float32)
    y_test = torch.tensor(test_labels, dtype=torch.long)
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    actual_batch_size = min(batch_size, len(labeled_indices))
    train_loader = DataLoader(train_data, batch_size=actual_batch_size, shuffle=True)
    linear_model = nn.Linear(feature_dim, num_classes).to(device)
    optimizer = optim.SGD(linear_model.parameters(), lr=lr, momentum=0.9, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.CrossEntropyLoss()
    linear_model.train()
    for epoch in range(num_epochs):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = linear_model(X_batch)
            loss = criterion(outputs, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
    linear_model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        test_data = torch.utils.data.TensorDataset(X_test.to(device), y_test.to(device))
        test_loader = DataLoader(test_data, batch_size=256, shuffle=False)
        for X_batch, y_batch in test_loader:
            outputs = linear_model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return 100.0 * correct / total

print("Linear classifier ready")


In [ ]:
# Framework (ii) — DNS Modified (RUN ON COLAB — fast)
BUDGET_PER_ROUND = 10
NUM_ROUNDS = 5
NUM_REPETITIONS = 10

dns_fw2_all = []

for rep in range(NUM_REPETITIONS):
    set_seed(rep)
    print(f"\nDNS Framework(ii) — Rep {rep+1}/{NUM_REPETITIONS}")
    labeled_indices = []
    round_accs = []

    for round_num in range(NUM_ROUNDS):
        new_indices = typicality_clustering_select_dns(
            features=features, budget=BUDGET_PER_ROUND,
            labeled_indices=labeled_indices, max_clusters=500
        )
        labeled_indices.extend(new_indices)
        acc = train_linear_classifier(
            labeled_indices, features, train_labels,
            test_features, test_labels, device
        )
        round_accs.append(acc)
        print(f"  Round {round_num+1}: Budget={len(labeled_indices)}, Acc={acc:.2f}%")

    dns_fw2_all.append(round_accs)

dns_fw2_all = np.array(dns_fw2_all)
dns_fw2_mean = dns_fw2_all.mean(axis=0)
dns_fw2_std = dns_fw2_all.std(axis=0)

print(f"\nDNS Framework (ii) Results:")
for i in range(NUM_ROUNDS):
    b = (i+1) * BUDGET_PER_ROUND
    print(f"Budget {b:4d}: {dns_fw2_mean[i]:.2f}% +/- {dns_fw2_std[i]:.2f}%")


---
## 9. Plot Results

Compare DNS vs Original (K=20) vs Random using hardcoded results from original notebook.


In [ ]:
# Hardcode results from original notebook — Framework (i)
mean_accuracies = np.array([14.57, 18.92, 19.71, 22.11, 22.53])
std_accuracies = np.array([0.81, 1.19, 0.98, 0.79, 0.62])
random_mean = np.array([14.43, 16.19, 17.81, 19.37, 20.72])
random_std = np.array([1.60, 3.14, 2.48, 2.81, 2.56])

budgets = [10, 20, 30, 40, 50]

plt.figure(figsize=(10, 6))
plt.plot(budgets, dns_fw1_mean, 'o-', label='TPC_RP (DNS)', color='red', linewidth=2)
plt.fill_between(budgets, dns_fw1_mean - dns_fw1_std, dns_fw1_mean + dns_fw1_std, alpha=0.2, color='red')
plt.plot(budgets, mean_accuracies, 's-', label='TPC_RP (K=20)', color='blue', linewidth=2)
plt.fill_between(budgets, mean_accuracies - std_accuracies, mean_accuracies + std_accuracies, alpha=0.2, color='blue')
plt.plot(budgets, random_mean, '^--', label='Random', color='gray', linewidth=2)
plt.fill_between(budgets, random_mean - random_std, random_mean + random_std, alpha=0.2, color='gray')
plt.xlabel('Cumulative Budget', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Task 3: DNS vs Original (K=20) — Framework (i) Fully Supervised', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('task3_dns_framework_i.png', dpi=150)
plt.show()


---
## 10. Results Summary Table


In [ ]:
print("=" * 70)
print(f"{'Budget':<10} {'DNS':<20} {'K=20 (Original)':<20} {'Random':<20}")
print("=" * 70)
for i in range(NUM_ROUNDS):
    b = (i+1) * BUDGET_PER_ROUND
    print(f"{b:<10} "
          f"{dns_fw1_mean[i]:5.2f}% \u00b1 {dns_fw1_std[i]:4.2f}%   "
          f"{mean_accuracies[i]:5.2f}% \u00b1 {std_accuracies[i]:4.2f}%   "
          f"{random_mean[i]:5.2f}% \u00b1 {random_std[i]:4.2f}%")
print("=" * 70)


---
## GitHub Repository

🔗 `https://github.com/YOUR_USERNAME/YOUR_REPO_NAME`
